# GBFS API exploration (Bergen city bikes)

This notebook shows:
- how to access the GBFS API
- what datasets are available
- which fields are most useful for our data pipeline (`Kafka -> Data Lake -> Warehouse -> Dashboard`)

In [5]:
import requests
import pandas as pd
from datetime import datetime, timezone

GBFS_ROOT_URL = "https://gbfs.urbansharing.com/bergenbysykkel.no/gbfs.json"


def fetch_json(url: str) -> dict:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.json()

In [8]:
gbfs_root = fetch_json(GBFS_ROOT_URL)
gbfs_root

{'last_updated': 1774290570,
 'ttl': 15,
 'version': '2.3',
 'data': {'nb': {'feeds': [{'name': 'system_information',
     'url': 'https://gbfs.urbansharing.com/bergenbysykkel.no/system_information.json'},
    {'name': 'vehicle_types',
     'url': 'https://gbfs.urbansharing.com/bergenbysykkel.no/vehicle_types.json'},
    {'name': 'system_pricing_plans',
     'url': 'https://gbfs.urbansharing.com/bergenbysykkel.no/system_pricing_plans.json'},
    {'name': 'station_information',
     'url': 'https://gbfs.urbansharing.com/bergenbysykkel.no/station_information.json'},
    {'name': 'station_status',
     'url': 'https://gbfs.urbansharing.com/bergenbysykkel.no/station_status.json'}]}}}

In [11]:
feeds = gbfs_root["data"]["nb"]["feeds"]

feeds_df = pd.DataFrame(feeds).sort_values("name").reset_index(drop=True)
feeds_df

,name,url
0,station_information,https://gbfs.urbansharing.com/bergenbysykkel.n...
1,station_status,https://gbfs.urbansharing.com/bergenbysykkel.n...
2,system_information,https://gbfs.urbansharing.com/bergenbysykkel.n...
3,system_pricing_plans,https://gbfs.urbansharing.com/bergenbysykkel.n...
4,vehicle_types,https://gbfs.urbansharing.com/bergenbysykkel.n...


## Most useful endpoints for this project

- `station_status` (real-time): bikes/docks availability and `last_reported` (best source for Kafka events)
- `station_information` (static-ish metadata): name, coordinates, capacity (best source for dimension table)
- `system_information` (context): operator/system metadata (optional)

In [22]:
feed_map = {item["name"]: item["url"] for item in feeds}
station_status_url = feed_map["station_status"]
station_info_url = feed_map["station_information"]
system_info_url = feed_map["system_information"]

station_status = fetch_json(station_status_url)
station_info = fetch_json(station_info_url)
system_info = fetch_json(system_info_url)

status_df = pd.DataFrame(station_status["data"]["stations"])
info_df = pd.DataFrame(station_info["data"]["stations"])

print("station_status_url:", station_status_url)
print("station_information_url:", station_info_url)
print("system_info_url:", system_info_url)
print()
print("status rows:", len(status_df), "| info rows:", len(info_df))
print("status columns:", sorted(status_df.columns)[:12], "...")
print("info columns:", sorted(info_df.columns)[:12], "...")

status_df.head(3)

station_status_url: https://gbfs.urbansharing.com/bergenbysykkel.no/station_status.json
station_information_url: https://gbfs.urbansharing.com/bergenbysykkel.no/station_information.json
system_info_url: https://gbfs.urbansharing.com/bergenbysykkel.no/system_information.json

status rows: 98 | info rows: 98
status columns: ['is_installed', 'is_renting', 'is_returning', 'last_reported', 'num_bikes_available', 'num_docks_available', 'num_vehicles_available', 'station_id', 'vehicle_types_available'] ...
info columns: ['address', 'capacity', 'cross_street', 'is_virtual_station', 'lat', 'lon', 'name', 'rental_uris', 'station_area', 'station_id'] ...


,station_id,is_installed,is_renting,is_returning,last_reported,num_vehicles_available,num_bikes_available,num_docks_available,vehicle_types_available
0,5356,True,True,True,1774291120,4,4,12,"[{'vehicle_type_id': 'bike', 'count': 4}]"
1,5255,True,True,True,1774291120,6,6,7,"[{'vehicle_type_id': 'bike', 'count': 6}]"
2,4963,True,True,True,1774291120,3,3,10,"[{'vehicle_type_id': 'bike', 'count': 3}]"


## Fields I'm gonna keep (useful for pipeline)

For **Kafka events** (`station_status`):
- `station_id`
- `num_bikes_available`
- `num_docks_available`
- `last_reported` (event timestamp)

In [27]:
status_df["last_reported"] = pd.to_datetime(
    status_df["last_reported"], unit="s", utc=True
).dt.tz_convert("Europe/Oslo")

status_df[[
    "station_id",
    "num_bikes_available",
    "num_docks_available",
    "last_reported",
]].head(10)

,station_id,num_bikes_available,num_docks_available,last_reported
0,5356,4,12,2026-03-23 19:38:40+01:00
1,5255,6,7,2026-03-23 19:38:40+01:00
2,4963,3,10,2026-03-23 19:38:40+01:00
3,4962,1,8,2026-03-23 19:38:40+01:00
4,3397,7,9,2026-03-23 19:38:40+01:00
5,3396,5,19,2026-03-23 19:38:40+01:00
6,2352,1,6,2026-03-23 19:38:40+01:00
7,2338,1,6,2026-03-23 19:38:40+01:00
8,2336,1,13,2026-03-23 19:38:40+01:00
9,2322,15,10,2026-03-23 19:38:40+01:00


For dimension table (`station_information`):
- `station_id`
- `name`
- `lat`, `lon`
- `capacity`

In [18]:
info_df[["station_id",
"name",
"lat","lon",
"capacity"
]].head(10)

,station_id,name,lat,lon,capacity
0,5356,Fløen Bybanestopp,60.381153,5.353251,16
1,5255,Tollbodalmenningen,60.399223,5.310526,13
2,4963,Kronstad Allmenning,60.368777,5.342295,13
3,4962,Mindemyren bybanestopp,60.365322,5.344795,9
4,3397,Kristianborg bybanestopp,60.357797,5.342070,16
5,3396,Kronstad bybanestopp,60.371787,5.349553,25
6,2352,Bontelabo 8,60.402481,5.316471,7
7,2338,Bontelabo,60.401758,5.316562,7
8,2336,Møllestranden,60.380044,5.350824,14
9,2322,Høyteknologisenteret,60.381750,5.331699,25


Optionally - `system_info table`:

In [25]:
system_info

{'last_updated': 1774291110,
 'ttl': 15,
 'version': '2.3',
 'data': {'system_id': 'bergen-city-bike',
  'language': 'nb',
  'name': 'Bergen Bysykkel',
  'operator': 'UIP Bauer Media Outdoor Norge AS',
  'timezone': 'Europe/Oslo',
  'phone_number': '90259737',
  'email': 'post@bergenbysykkel.no',
  'rental_apps': {'android': {'discovery_uri': 'bergenbysykkel://',
    'store_uri': 'https://play.google.com/store/apps/details?id=com.urbansharing.citybike.bergen&hl=en'},
   'ios': {'discovery_uri': 'bergenbysykkel://',
    'store_uri': 'https://itunes.apple.com/us/app/bergen-bysykkel/id1242856965?ls=1&mt=8'}}}}